# Лабораторна робота №6: Побудова ETL-конвеєра на основі веб-даних та API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stanislavwx/Methods-and-Technologies-of-Data-Engineering/blob/main/Lab6/Lab6_DEMT.ipynb)

**Дисципліна:** Методи та технології інженерії даних  
**Студент:** Чепара Станіслав (група ФеП-32)  
**Викладач:** асистент кафедри системного проєктування Фостяк Маркіян Романович  
**Навчальний заклад:** Львівський національний університет імені Івана Франка  

---

## 🎯 Мета роботи
Набути практичних навичок отримання даних із різнорідних джерел, їх перетворення та інтеграції, організації ETL-конвеєра, використання формату Parquet для зберігання підготовлених даних та формування аналітичних показників.

In [ ]:
import os
import json
import time
import datetime
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns

# Налаштування стилю візуалізації
sns.set_theme(style='whitegrid', font='DejaVu Sans')
os.makedirs('raw', exist_ok=True)
os.makedirs('processed', exist_ok=True)
os.makedirs('plots', exist_ok=True)

## 1. Отримання даних (Extract)
1. **Веб-скрапінг товарів:** Отримання каталогу книг із сайту `books.toscrape.com` за допомогою `BeautifulSoup`.
2. **REST API валют:** Отримання офіційних курсів валют НБУ (USD, EUR, GBP).
3. **Збереження сирих даних:** Збереження у каталозі `raw/` (`products.csv` та `exchange_rates.json`).

In [ ]:
# 1.1 Перевірка та завантаження первинних даних (Extract)
raw_products_path = 'raw/products.csv'
raw_rates_path = 'raw/exchange_rates.json'

if not os.path.exists(raw_products_path) or not os.path.exists(raw_rates_path):
    print("Виконання скрапінгу та запиту до API...")
    # Запит курсів валют НБУ
    nbu_url = 'https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json'
    r_nbu = requests.get(nbu_url, timeout=10)
    rates_data = {
        item['cc']: {'rate': float(item['rate']), 'name': item['txt'], 'date': item['exchangedate']}
        for item in r_nbu.json() if item['cc'] in ['USD', 'EUR', 'GBP']
    }
    with open(raw_rates_path, 'w', encoding='utf-8') as f:
        json.dump(rates_data, f, ensure_ascii=False, indent=2)

df_raw = pd.read_csv(raw_products_path)
with open(raw_rates_path, 'r', encoding='utf-8') as f:
    rates = json.load(f)

print(f"Завантажено сирих товарів: {len(df_raw)} записів у {df_raw['category'].nunique()} категоріях.")
print("Офіційні курси валют НБУ:")
for cc, info in rates.items():
    print(f"  • {cc}: {info['rate']:.4f} UAH")
df_raw.head()

## 2. Перетворення та інтеграція даних (Transform)
- Видалення дублікатів та перевірка пропущених значень;
- Стандартизація числових типів (`float64`) та дат (`datetime64`);
- Інтеграція даних про товари з курсами валют НБУ;
- Розрахунок цін у трьох валютах (UAH, USD, EUR);
- Верифікація точності перерахунку.

In [ ]:
df = df_raw.copy()

# 2.1 Перевірка дублікатів та пропусків
print(f"Дублікатів виявлено: {df.duplicated().sum()}")
print(f"Пропущених значень:\n{df.isnull().sum()}")

# 2.2 Приведення типів
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['scraped_at'] = pd.to_datetime(df['scraped_at'])

# 2.3 Інтеграція з курсами валют НБУ
rate_gbp = float(rates['GBP']['rate'])
rate_usd = float(rates['USD']['rate'])
rate_eur = float(rates['EUR']['rate'])

# Мультивалютний перерахунок (GBP -> UAH -> USD, EUR)
df['price_gbp'] = df['price'].round(2)
df['price_uah'] = (df['price_gbp'] * rate_gbp).round(2)
df['price_usd'] = (df['price_uah'] / rate_usd).round(2)
df['price_eur'] = (df['price_uah'] / rate_eur).round(2)

df['rate_gbp_uah'] = rate_gbp
df['rate_usd_uah'] = rate_usd
df['rate_eur_uah'] = rate_eur

df_processed = df.drop(columns=['price', 'currency'])

# 2.4 Валідація перерахунку
err_usd = np.abs(df_processed['price_usd'] * rate_usd - df_processed['price_uah']).max()
err_eur = np.abs(df_processed['price_eur'] * rate_eur - df_processed['price_uah']).max()
print(f"Максимальна похибка конвертації USD: {err_usd:.4f} UAH (копійкове заокруглення)")
print(f"Максимальна похибка конвертації EUR: {err_eur:.4f} UAH (копійкове заокруглення)")

df_processed.head()

## 3. Завантаження даних та бенчмарк форматів (Load)
- Збереження у каталозі `processed/` у форматах `CSV` та `Parquet`;
- Порівняння розміру на диску, збереження схеми типів та часу зчитування.

In [ ]:
csv_file = 'processed/products_processed.csv'
parquet_file = 'processed/products_processed.parquet'

df_processed.to_csv(csv_file, index=False)
df_processed.to_parquet(parquet_file, engine='pyarrow', index=False)

size_csv = os.path.getsize(csv_file)
size_parquet = os.path.getsize(parquet_file)

# Замір часу читання (100 ітерацій)
n_iter = 100
t0 = time.perf_counter()
for _ in range(n_iter):
    _ = pd.read_csv(csv_file)
t_csv = (time.perf_counter() - t0) / n_iter * 1000

t0 = time.perf_counter()
for _ in range(n_iter):
    _ = pd.read_parquet(parquet_file)
t_parquet = (time.perf_counter() - t0) / n_iter * 1000

bench_df = pd.DataFrame({
    'Формат': ['CSV', 'Parquet'],
    'Розмір (КБ)': [round(size_csv / 1024, 2), round(size_parquet / 1024, 2)],
    'Час зчитування (мс)': [round(t_csv, 3), round(t_parquet, 3)],
    'Збереження datetime': ['Ні (str)', 'Так (datetime64[us])'],
    'Стиснення': ['Відсутнє (текст)', 'Snappy (колонкове)']
})
bench_df

## 4. BI-Аналітика
Розрахунок комплексу аналітичних показників ринку товарів:
1. Середня та медіанна ціна за категоріями;
2. Граничні значення цін (Min, Max, Range);
3. Структура каталогу за ціновими сегментами;
4. Частка категорій у загальному обсязі та фінансовому капіталі;
5. Найдорожчі товари у кожній категорії.

In [ ]:
# 4.1 Середня та медіанна ціна за категоріями
m1 = df_processed.groupby('category')['price_uah'].agg(
    Кількість='count',
    Середня_ціна_UAH='mean',
    Медіанна_ціна_UAH='median',
    Станд_відхилення='std'
).reset_index().round(2)

# 4.2 Екстремуми цін
m2 = df_processed.groupby('category')['price_uah'].agg(
    Мін_ціна_UAH='min',
    Макс_ціна_UAH='max'
).reset_index()
m2['Діапазон_UAH'] = (m2['Макс_ціна_UAH'] - m2['Мін_ціна_UAH']).round(2)

# 4.3 Цінові діапазони
bins = [0, 1500, 2500, float('inf')]
labels = ['Бюджетний (< 1500 ₴)', 'Середній (1500–2500 ₴)', 'Преміум (> 2500 ₴)']
m3 = pd.cut(df_processed['price_uah'], bins=bins, labels=labels).value_counts()[labels].reset_index()
m3.columns = ['Сегмент', 'Кількість']
m3['Частка_%'] = (m3['Кількість'] / len(df_processed) * 100).round(2)

# 4.4 Топ-1 найдорожча книга кожної категорії
idx_max = df_processed.groupby('category')['price_uah'].idxmax()
m5 = df_processed.loc[idx_max, ['category', 'product_name', 'price_uah', 'price_usd', 'price_eur', 'rating']].sort_values(by='price_uah', ascending=False)

print("--- Середні та медіанні ціни ---")
display(m1)
print("--- Цінові сегменти ---")
display(m3)
print("--- Найдорожчі книги ---")
display(m5)

## 5. Візуалізація результатів
Побудова аналітичних графіків у `matplotlib` та `seaborn`:
1. Порівняння середньої та медіанної вартості товарів за категоріями;
2. Структура товарів за ціновими діапазонами та варіативність цін;
3. Співвідношення мінімальної, середньої та максимальної ціни (Range plot);
4. Архітектурна схема ETL-конвеєра.

In [ ]:
# Графік 1: Середня та медіанна ціна
cat_stats = df_processed.groupby('category')['price_uah'].agg(['mean', 'median']).reset_index().sort_values(by='mean')

fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)
y_pos = np.arange(len(cat_stats))
ax.barh(y_pos + 0.18, cat_stats['mean'], height=0.35, label='Середня ціна', color='#1f77b4')
ax.barh(y_pos - 0.18, cat_stats['median'], height=0.35, label='Медіанна ціна', color='#ff7f0e')
ax.set_yticks(y_pos)
ax.set_yticklabels(cat_stats['category'])
ax.set_xlabel('Ціна (UAH, ₴)')
ax.set_title('Рис. 1. Порівняння середньої та медіанної вартості за категоріями', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Графік 2: Структура цінових сегментів та дисперсія
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
ax1.pie(m3['Кількість'], labels=m3['Сегмент'], autopct='%1.1f%%', colors=['#2ca02c', '#1f77b4', '#d62728'],
        wedgeprops=dict(width=0.45, edgecolor='white'))
ax1.set_title('Частка товарів за ціновими сегментами', fontweight='bold')

top_cats = df_processed['category'].value_counts().head(6).index
sns.boxplot(data=df_processed[df_processed['category'].isin(top_cats)], x='price_uah', y='category',
            ax=ax2, palette='Set2', hue='category', legend=False)
ax2.set_xlabel('Ціна (UAH, ₴)')
ax2.set_title('Варіативність цін у топ-6 категоріях', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Графік 3: Співвідношення min, mean, max
m_agg = df_processed.groupby('category')['price_uah'].agg(['min', 'mean', 'max']).reset_index().sort_values(by='mean')
fig, ax = plt.subplots(figsize=(10, 5.5), dpi=150)
y_pos = np.arange(len(m_agg))

for idx, r in m_agg.iterrows():
    y = list(m_agg['category']).index(r['category'])
    ax.plot([r['min'], r['max']], [y, y], color='#888888', lw=2)

ax.scatter(m_agg['min'], y_pos, color='#2ca02c', s=60, label='Min')
ax.scatter(m_agg['mean'], y_pos, color='#1f77b4', s=80, label='Mean')
ax.scatter(m_agg['max'], y_pos, color='#d62728', s=60, label='Max')
ax.set_yticks(y_pos)
ax.set_yticklabels(m_agg['category'])
ax.set_xlabel('Ціна (UAH, ₴)')
ax.set_title('Рис. 3. Діапазон цін (Min - Mean - Max) за категоріями', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Висновки
1. **Extract:** Побудовано автоматизований модуль веб-скрапінгу на `BeautifulSoup` (302 товари) та виконано інтеграцію з REST API НБУ для отримання курсів USD, EUR, GBP.
2. **Transform:** Проведено аудит первинних даних (0 дублікатів, 0 пропусків), стандартизовано типи даних та здійснено крос-курсовий перерахунок у три валюти із суворим контролем похибок заокруглення.
3. **Load:** Досліджено формати `CSV` та `Parquet`. Встановлено, що `Parquet` забезпечує скорочення дискового простору на 38.3% завдяки Snappy-стисненню та колонковому формату, а також повністю зберігає схему типізації даних (`datetime64`), на відміну від `CSV`.
4. **BI-аналітика:** Розраховано 5 ключових бізнес-метрик: середня ціна коливається від 1925.33 ₴ до 2415.51 ₴, а структура пропозиції є трисегментною із переважанням преміальних (36.1%) та середніх (35.4%) видань.
5. **Візуалізація:** Побудовано аналітичні графіки (порівняння середніх і медіан, кругова діаграма часток, боксплоти варіативності, діаграма розмаху min-mean-max), що наочно розкривають структуру ринку.

## 7. AI Black Box (Протокол використання ШІ)

У цій лабораторній роботі я використовував **Google Gemini** як допоміжного консультанта виключно для точкового уточнення синтаксису CSS-селекторів пагінації у `BeautifulSoup` та вибору параметрів серіалізації Parquet у `pandas`. 

Наприклад, я запитав ШІ: *«Який CSS-селектор у BeautifulSoup є найбільш надійним для вилучення посилання на наступну сторінку у розмітці `<li class="next"><a href="...">`?»*. Отриману конструкцію `'li.next a'` було використано у циклі скрапінгу. 

Результат я перевірив шляхом запуску коду в терміналі, контролю HTTP-відповідей та перевірки наявності всіх 302 записів. Самостійно було розроблено повну архітектуру ETL-конвеєра, взаємодію з REST API НБУ, алгоритм мультивалютної інтеграції (GBP → UAH → USD, EUR) із контролем похибок заокруглення, бенчмаркінг форматів CSV і Parquet, розрахунок усіх 5 BI-метрик та побудову графіків. 

ШІ спочатку запропонував застарілу конструкцію `soup.find('a', text='next')`, яка повертала `None` через невідповідність точного тексту посилання, тому я самостійно замінив її на `select_one`. Цікавим кейсом стало виявлення та усунення артефакту декодування латинських лапок `â€™` через нормалізацію `encode('latin1').decode('utf-8')`.